# OMNI-902 · UCF-Crime's clip vectors for the forecasting model, InternVideo2 (Colab GPU)

The PFM reads InternVideo2 clip vectors (D-53). Runs the repo's own code from a clone of the branch:
`scripts/encode_pfm_windows.py` (the grid and the runner) with ai_workers' InternVideo2 and window functions.

**What it makes:** for every window of `datasets/windows/ucf_crime.csv` (1,448 videos, 2,896 windows: 30 s and
60 s, every split), a clip vector every **1 s** from the window's last frame back to the window's start (D-54; no
footage before the window is needed, D-55): **82,777 clips**. Each clip is Algorithm 2's (8 frames from the 1 s
before `t`). Videos are read straight from Drive, 8 at a time ahead of the GPU, and decoding stops at the last frame
a window needs. One file per video on Drive, so a lost session resumes where it stopped.

1. **Setup:** clone, install the pinned versions, Drive
2. **InternVideo2** → `MyDrive/omnisight/embeddings/pfm/ucf_crime/internvideo2/`
3. **Check and save:** every video done, the time per clip → `MyDrive/omnisight/results/omni902_ucf_encoding.json`

Runtime → Change runtime type → **GPU** (A100: about 4.5 h; L4: about 6.5 h). Then Run all. If the session ends,
Run all again: finished videos are skipped.

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

# The versions in uv.lock; Colab keeps its own GPU build of torch and torchvision.
!pip install -q "transformers==5.20.1" "huggingface-hub==2.2.0" "safetensors==0.8.0" "open-clip-torch==3.3.0" -e /content/omni-sight/services/common

sys.path.insert(0, "/content/omni-sight/services/common")
sys.path.insert(0, "/content/omni-sight/services/ai_workers")
sys.path.insert(0, "/content/omni-sight/scripts")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2 · InternVideo2 (the product's pin, D-33, D-53), float32
import json, time
from pathlib import Path
import encode_pfm_windows as pfm
from omnisight_common.settings import CommonSettings
from app.encoder import ensure_checkpoint
from app import clip_encoder
from app.clip_encoder import ClipTextEncoder

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
UCF = DRIVE / "data" / "ucf_crime"
OUT = DRIVE / "embeddings" / "pfm" / "ucf_crime" / "internvideo2"
WINDOWS = pfm.read_windows(REPO_DIR / "datasets" / "windows" / "ucf_crime.csv")
CLIPS = sum(len(pfm.grid_frames(rows, float(rows[0]["fps"]), pfm.STEPS_S["ucf_crime"], context_s=0.0)) for rows in WINDOWS.values())
print(f"{len(WINDOWS):,} videos, {CLIPS:,} grid clips")
assert len(WINDOWS) == 1448

SESSION = time.strftime("%Y-%m-%d %H:%M") + " " + torch.cuda.get_device_name(0)
pins = {name: CommonSettings.model_fields[name].default for name in ("clip_model_id", "clip_model_revision")}
model_dir = ensure_checkpoint(pins["clip_model_id"], pins["clip_model_revision"], DRIVE / "models" / "pretrained" / "internvideo2-clip-s", clip_encoder.ALLOW_PATTERNS)
encoder = ClipTextEncoder(model_dir, device="cuda")
manifest = pfm.run(WINDOWS, UCF, OUT, encoder, clip_encoder, pins, pfm.STEPS_S["ucf_crime"], context_s=0.0, session=SESSION)
print("failed:", manifest["failed"])

In [ ]:
# 3 · Check and save the summary
import numpy as np
s = pfm.summary(WINDOWS, OUT)
print(f"{s['done']} of {s['videos']} videos, {s['clips']:,} clips, {s['ms_per_clip']} ms per clip in the encoder, "
      f"{s['encoder_seconds'] / 3600:.2f} h; failed {len(s['failed'])}: {list(s['failed'].items())[:5]}")
frames = 0
for key in WINDOWS:
    if pfm.out_file(OUT, key).is_file():
        with np.load(pfm.out_file(OUT, key)) as f:
            frames += int(f["frames_read"])
s["frames_decoded"] = frames
out = DRIVE / "results" / "omni902_ucf_encoding.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"commit": COMMIT, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
                           "clips_expected": CLIPS, "internvideo2": s}, indent=1))
print(out)
assert s["done"] == 1448 and s["clips"] == CLIPS and not s["failed"]